
# Micro-proyecto1 · Fine-Tuning **ResNet‑50** (Skeleton)
**Autor:** _Tu equipo_  

**Fecha:** 2025-10-27 01:06

> Este cuaderno es un **esqueleto** listo para completar el fine-tuning de **ResNet‑50** sobre un dataset de **MRI cerebrales** en **escala de grises** (4 clases). Cumple principios **SOLID**, **DRY** y una organización inspirada en **Atomic Design**:
- **Átomos**: transformaciones, métricas, pérdidas, optimizadores
- **Moléculas**: dataset y *data loaders*
- **Organismos**: modelo y *trainer*
- **Página/Escena**: `main()` y ciclo de entrenamiento/validación configurable

> **Nota de diseño**: por requerimiento, **no** aplicamos rotaciones inicialmente. Más adelante podrás activarlas para evaluar impacto.


> **Nota de datos (Opción B):** Este cuaderno realiza una partición **estratificada 80/10/10** de `./data_set` **sin mover archivos**. Usamos `ImageFolder` como raíz (clases por carpeta), obtenemos índices con **StratifiedShuffleSplit** y construimos **Subsets** con transforms por split para `train`, `val` y `test`. De esta forma, el **pipeline** de preprocesamiento queda contenido, reproducible (semillas) y sin duplicar datos ni lógica.

## 1. Librerías

In [1]:
# Todas las importaciones en un solo lugar (principio: alta cohesión, separación de intereses)
from __future__ import annotations
import os
import math
import time
import copy
import random
from dataclasses import dataclass, asdict
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import torch
from torch import nn, optim
from torch.optim import Optimizer
from torch.optim.lr_scheduler import OneCycleLR, ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import StratifiedShuffleSplit  # <-- estratificación 80/10/10

# Opcional: para LR Finder (visualización)
import matplotlib.pyplot as plt



## 2. Hiperparámetros (todas las "perillas" en un solo sitio)
> Ajusta desde aquí todo lo relevante: rutas, *batch size*, *learning rate*, *scheduler*, *augmentations*, etc.


In [2]:
@dataclass
class Config:
    # --- Rutas ---
    data_dir: str = "./data_set"      # raíz plana con subcarpetas por clase (ImageFolder)
    out_dir: str = "./outputs"
    run_name: str = "resnet50_grayscale_ft"
    # --- Datos ---
    img_size: int = 224
    num_classes: int = 4
    input_mode: str = "replicate_to_3"  # 'replicate_to_3' | 'single_channel_conv'
    # --- Splits (Opción B: estratificados sin mover archivos) ---
    val_split: float = 0.10
    test_split: float = 0.10
    use_stratified_splits: bool = True
    sampler_seed: int = 42
    # --- Entrenamiento ---
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    epochs: int = 20
    batch_size: int = 32
    base_lr: float = 3e-4
    weight_decay: float = 1e-4
    momentum: float = 0.9
    optimizer_name: str = "adamw"  # 'adamw' | 'sgd'
    # --- Scheduler ---
    scheduler_name: str = "reduce_on_plateau"  # 'one_cycle' | 'reduce_on_plateau' | 'none'
    one_cycle_max_lr: float = 1e-3
    plateau_factor: float = 0.5
    plateau_patience: int = 2
    # --- Early Stopping ---
    early_stopping_patience: int = 5
    # --- DataLoader ---
    num_workers: int = 4
    pin_memory: bool = True

cfg = Config()
print("Config:", asdict(cfg))


Config: {'data_dir': './data_set', 'out_dir': './outputs', 'run_name': 'resnet50_grayscale_ft', 'img_size': 224, 'num_classes': 4, 'input_mode': 'replicate_to_3', 'val_split': 0.1, 'test_split': 0.1, 'use_stratified_splits': True, 'sampler_seed': 42, 'seed': 42, 'device': 'cpu', 'epochs': 20, 'batch_size': 32, 'base_lr': 0.0003, 'weight_decay': 0.0001, 'momentum': 0.9, 'optimizer_name': 'adamw', 'scheduler_name': 'reduce_on_plateau', 'one_cycle_max_lr': 0.001, 'plateau_factor': 0.5, 'plateau_patience': 2, 'early_stopping_patience': 5, 'num_workers': 4, 'pin_memory': True}


## 3. Utilidades: semilla y helpers

In [3]:

def set_seed(seed: int) -> None:
    """Fija todas las semillas relevantes para reproducibilidad."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(cfg.seed)
os.makedirs(cfg.out_dir, exist_ok=True)



## 4. Átomos: **Transforms** (sin rotaciones por ahora)
> Mantener el preprocesamiento y *augmentations* con mínima sorpresa.  
> **No** se aplican rotaciones inicialmente (requisito).


In [4]:

class GrayTo3Channels(nn.Module):
    """Convierte tensores [1,H,W] a [3,H,W] replicando el canal gris (DRY: reutilizable)."""
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 3 and x.size(0) == 1:
            return x.repeat(3, 1, 1)
        return x

def build_transforms(cfg: Config) -> Tuple[transforms.Compose, transforms.Compose]:
    """Crea transforms de train/valid según 'input_mode'. Sin rotaciones por ahora."""
    common = [
        transforms.Grayscale(num_output_channels=1),  # Asegura 1 canal de entrada
        transforms.Resize((cfg.img_size, cfg.img_size)),
        # Augmentations leves (sin rotación): flips horizontales opcionales y jitter moderado
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1)  # prudente en MRI
    ]
    to_tensor = [transforms.ToTensor()]

    if cfg.input_mode == "replicate_to_3":
        post = [GrayTo3Channels(), transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.25,0.25,0.25])]
    else:  # 'single_channel_conv'
        post = [transforms.Normalize(mean=[0.5], std=[0.25])]

    train_tfms = transforms.Compose(common + to_tensor + post)
    valid_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor()
    ] + ( [GrayTo3Channels(), transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.25,0.25,0.25])] if cfg.input_mode == "replicate_to_3"
          else [transforms.Normalize(mean=[0.5], std=[0.25])] ))
    return train_tfms, valid_tfms



## 5. Moléculas: **Dataset/DataModule**
- Se usa `ImageFolder` (estructura `root/class_x/*.jpg`).
- Si no tienes split, puedes duplicar el directorio y dejar `train/` y `val/` o, alternativamente, hacer un **split aleatorio**.


In [5]:
class DataModule:
    """Carga de datos con **splits estratificados 80/10/10** sin mover archivos.
    - Usa un único ImageFolder raíz para obtener 'samples' y 'targets'.
    - Crea índices con StratifiedShuffleSplit.
    - Construye tres Subsets con transforms por split: train/val/test.
    """
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.class_to_idx = None
        self.idx_to_class = None
        self.train_tfms, self.valid_tfms = build_transforms(cfg)

    def _collect_targets(self):
        base = datasets.ImageFolder(self.cfg.data_dir)  # sin transforms, solo para metadata
        y = base.targets  # lista de enteros
        self.class_to_idx = base.class_to_idx
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        return base, y

    def setup(self) -> None:
        base, y = self._collect_targets()
        n_total = len(y)
        # Paso 1: test 10%
        sss1 = StratifiedShuffleSplit(n_splits=1, test_size=self.cfg.test_split, random_state=self.cfg.sampler_seed)
        all_idx = np.arange(n_total)
        trainval_idx, test_idx = next(sss1.split(all_idx, y))

        # Paso 2: val del remanente, tamaño proporcional para alcanzar 10% del total
        val_portion = self.cfg.val_split / (1.0 - self.cfg.test_split)
        y_trainval = [y[i] for i in trainval_idx]
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_portion, random_state=self.cfg.sampler_seed)
        train_idx_rel, val_idx_rel = next(sss2.split(trainval_idx, y_trainval))
        train_idx = trainval_idx[train_idx_rel]
        val_idx   = trainval_idx[val_idx_rel]

        # DataSets base con transforms por split
        train_base = datasets.ImageFolder(self.cfg.data_dir, transform=self.train_tfms)
        val_base   = datasets.ImageFolder(self.cfg.data_dir, transform=self.valid_tfms)
        test_base  = datasets.ImageFolder(self.cfg.data_dir, transform=self.valid_tfms)

        # Subsets por índices
        train_ds = torch.utils.data.Subset(train_base, train_idx.tolist())
        val_ds   = torch.utils.data.Subset(val_base,   val_idx.tolist())
        test_ds  = torch.utils.data.Subset(test_base,  test_idx.tolist())

        # Loaders
        self.train_loader = DataLoader(
            train_ds, batch_size=self.cfg.batch_size, shuffle=True,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory
        )
        self.val_loader = DataLoader(
            val_ds, batch_size=self.cfg.batch_size, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory
        )
        self.test_loader = DataLoader(
            test_ds, batch_size=self.cfg.batch_size, shuffle=False,
            num_workers=self.cfg.num_workers, pin_memory=self.cfg.pin_memory
        )

    def loaders(self) -> Tuple[DataLoader, DataLoader, DataLoader]:
        return self.train_loader, self.val_loader, self.test_loader



## 6. Organismos: **Modelo (ResNet‑50)**
**Grayscale:** dos opciones equivalentes funcionalmente:
1) **Replicar** canal gris → 3 canales (no modifica pesos originales).  
2) **Modificar** `conv1` para aceptar `in_channels=1` (promediar pesos preentrenados en RGB).

> Por defecto dejamos `replicate_to_3` (más simple y menos propenso a errores); puedes cambiar a `single_channel_conv` en `cfg`.


In [6]:

def adapt_first_conv_to_single_channel(model: nn.Module) -> nn.Module:
    """Convierte conv1 de ResNet-50 para aceptar 1 canal, promediando pesos RGB preentrenados."""
    conv1 = model.conv1
    if conv1.in_channels == 1:
        return model  # ya adaptado
    w = conv1.weight.data
    w_mean = w.mean(dim=1, keepdim=True)  # [64,1,7,7]
    new_conv1 = nn.Conv2d(
        in_channels=1, out_channels=conv1.out_channels,
        kernel_size=conv1.kernel_size, stride=conv1.stride,
        padding=conv1.padding, bias=conv1.bias is not None
    )
    with torch.no_grad():
        new_conv1.weight = nn.Parameter(w_mean)
        if conv1.bias is not None:
            new_conv1.bias = conv1.bias
    model.conv1 = new_conv1
    return model

class ModelBuilder:
    """Crea el modelo y su cabeza de clasificación (OCP: abierta a extensión)."""
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def build(self) -> nn.Module:
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        if self.cfg.input_mode == "single_channel_conv":
            model = adapt_first_conv_to_single_channel(model)

        # Reemplazar la cabeza (fc) por una capa para 4 clases
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, self.cfg.num_classes)
        return model



## 7. Átomos: **Pérdida, Métricas, Optimizador y Scheduler**
- Métrica primaria sugerida: **macro-F1** (útil ante desbalance).
- Implementamos `accuracy` y `macro_f1` básicos sin dependencias externas.


In [7]:

def criterion_fn() -> nn.Module:
    return nn.CrossEntropyLoss()

def accuracy(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    preds = y_pred.argmax(dim=1)
    return (preds == y_true).float().mean().item()

def macro_f1(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int) -> float:
    preds = y_pred.argmax(dim=1)
    f1_scores = []
    for c in range(num_classes):
        tp = ((preds == c) & (y_true == c)).sum().item()
        fp = ((preds == c) & (y_true != c)).sum().item()
        fn = ((preds != c) & (y_true == c)).sum().item()
        if tp + fp == 0 or tp + fn == 0:
            f1_scores.append(0.0)
        else:
            precision = tp / (tp + fp + 1e-12)
            recall    = tp / (tp + fn + 1e-12)
            if precision + recall == 0:
                f1_scores.append(0.0)
            else:
                f1_scores.append(2 * precision * recall / (precision + recall))
    return float(np.mean(f1_scores))

def build_optimizer(model: nn.Module, cfg: Config) -> Optimizer:
    if cfg.optimizer_name.lower() == "sgd":
        return optim.SGD(model.parameters(), lr=cfg.base_lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)
    else:
        return optim.AdamW(model.parameters(), lr=cfg.base_lr, weight_decay=cfg.weight_decay)

def build_scheduler(optimizer: Optimizer, cfg: Config, steps_per_epoch: Optional[int] = None):
    if cfg.scheduler_name == "one_cycle":
        assert steps_per_epoch is not None, "OneCycleLR requiere steps_per_epoch"
        return OneCycleLR(optimizer, max_lr=cfg.one_cycle_max_lr,
                          epochs=cfg.epochs, steps_per_epoch=steps_per_epoch)
    elif cfg.scheduler_name == "reduce_on_plateau":
        return ReduceLROnPlateau(optimizer, mode='max', factor=cfg.plateau_factor,
                                 patience=cfg.plateau_patience, verbose=True)
    else:
        return None



## 8. **LR Finder** (opcional, recomendado)
**Buena práctica** para elegir *learning rate*: ejecutar un **LR Range Test** (Leslie Smith).  
Procedimiento:
1. Fija el modelo en modo entrenamiento y comienza con un LR muy bajo (p.ej. `1e-7`).  
2. Incrementa el LR exponencialmente cada iteración hasta un valor alto (p.ej. `1`).  
3. Registra pérdida vs. LR (log-scale).  
4. Elige un valor en la región **antes** de que la pérdida comience a dispararse (o el valle de pérdida si es claro).

Alternativas/Complementos:
- **OneCycleLR** con `max_lr` elegido desde el LR Finder.  
- Mini *grid search* de LR en un **subset** del entrenamiento (rápido).

> Este bloque implementa el LR Range Test; úsalo con un *subset* para hacerlo ágil.


In [8]:

@torch.no_grad()
def plot_lr_finder(lrs: List[float], losses: List[float]) -> None:
    plt.figure()
    plt.plot(lrs, losses)
    plt.xscale('log')
    plt.xlabel('Learning Rate (log scale)')
    plt.ylabel('Loss')
    plt.title('LR Range Test')
    plt.show()

def lr_range_test(model: nn.Module, dataloader: DataLoader, cfg: Config,
                  start_lr: float = 1e-7, end_lr: float = 1.0, num_iters: int = 200) -> Tuple[List[float], List[float]]:
    model = copy.deepcopy(model).to(cfg.device)
    model.train()
    criterion = criterion_fn()
    optimizer = optim.AdamW(model.parameters(), lr=start_lr, weight_decay=cfg.weight_decay)

    lrs, losses = [], []
    lr_mult = (end_lr / start_lr) ** (1 / max(1, num_iters-1))
    lr = start_lr
    best_loss = float('inf')
    i = 0

    for images, targets in dataloader:
        images, targets = images.to(cfg.device), targets.to(cfg.device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        lrs.append(lr)
        losses.append(loss.item())

        # update LR
        lr *= lr_mult
        for g in optimizer.param_groups:
            g['lr'] = lr

        i += 1
        if i >= num_iters or lr > end_lr or loss.item() > 4 * best_loss:
            break
        best_loss = min(best_loss, loss.item())
    return lrs, losses



## 9. Organismos: **Trainer** (entrenamiento/validación, early stopping, checkpoints)
> Diseñado con SRP y extensión simple para *callbacks*.


In [9]:

class Trainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.best_score = -float('inf')
        self.no_improve_epochs = 0
        self.history = {'train_loss':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}

    def train_one_epoch(self, model, loader, criterion, optimizer):
        model.train()
        running_loss = 0.0
        for images, targets in loader:
            images, targets = images.to(self.cfg.device), targets.to(self.cfg.device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        return running_loss / len(loader.dataset)

    @torch.no_grad()
    def validate(self, model, loader, criterion):
        model.eval()
        running_loss = 0.0
        all_preds, all_targets = [], []
        for images, targets in loader:
            images, targets = images.to(self.cfg.device), targets.to(self.cfg.device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * images.size(0)
            all_preds.append(outputs.cpu())
            all_targets.append(targets.cpu())
        all_preds = torch.cat(all_preds, dim=0)
        all_targets = torch.cat(all_targets, dim=0)
        val_loss = running_loss / len(loader.dataset)
        val_acc = accuracy(all_targets, all_preds)
        val_f1  = macro_f1(all_targets, all_preds, self.cfg.num_classes)
        return val_loss, val_acc, val_f1

    def fit(self, model, train_loader, val_loader):
        model = model.to(self.cfg.device)
        criterion = criterion_fn()
        optimizer = build_optimizer(model, self.cfg)

        steps_per_epoch = len(train_loader)
        scheduler = build_scheduler(optimizer, self.cfg, steps_per_epoch=steps_per_epoch)

        best_weights = copy.deepcopy(model.state_dict())
        for epoch in range(self.cfg.epochs):
            train_loss = self.train_one_epoch(model, train_loader, criterion, optimizer)
            val_loss, val_acc, val_f1 = self.validate(model, val_loader, criterion)

            if isinstance(scheduler, OneCycleLR):
                scheduler.step()
            elif isinstance(scheduler, ReduceLROnPlateau):
                scheduler.step(val_f1)

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['val_f1'].append(val_f1)

            print(f"Epoch {epoch+1:03d}/{self.cfg.epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}")

            # Early stopping por macro-F1
            if val_f1 > self.best_score:
                self.best_score = val_f1
                best_weights = copy.deepcopy(model.state_dict())
                self.no_improve_epochs = 0
                torch.save(best_weights, os.path.join(self.cfg.out_dir, f"{self.cfg.run_name}_best.pth"))
            else:
                self.no_improve_epochs += 1
                if self.no_improve_epochs >= self.cfg.early_stopping_patience:
                    print("Early stopping activado.")
                    break
        model.load_state_dict(best_weights)
        return model, self.history


Celda de diagnóstico

Esta celda:

Llama a DataModule.setup()

Imprime tamaños de train/val/test

Verifica que la suma coincide con el total

Muestra distribución por clase en cada split

Confirma forma del primer batch (para chequear 1 canal vs 3 canales según input_mode)

In [ ]:
from collections import Counter
import numpy as np

# 1) Instanciar y preparar
dm = DataModule(cfg)
dm.setup()
train_loader, val_loader, test_loader = dm.loaders()

# 2) Tamaños por split
n_train = len(train_loader.dataset)
n_val   = len(val_loader.dataset)
n_test  = len(test_loader.dataset)
print(f"Sizes -> train: {n_train} | val: {n_val} | test: {n_test} | total: {n_train+n_val+n_test}")

# 3) Verificación de consistencia
#   Cada dataset es un Subset(ImageFolder), así que podemos acceder a sus indices y a targets del ImageFolder base.
def split_counts(subset):
    idxs = np.array(subset.indices)
    targets = np.array(subset.dataset.targets)
    labels = targets[idxs]
    return Counter(labels)

def pretty_counts(counter, idx_to_class):
    return { idx_to_class[k]: v for k, v in sorted(counter.items(), key=lambda kv: kv[0]) }

train_counts = split_counts(train_loader.dataset)
val_counts   = split_counts(val_loader.dataset)
test_counts  = split_counts(test_loader.dataset)

print("Class mapping:", dm.idx_to_class)
print("Train dist.:", pretty_counts(train_counts, dm.idx_to_class))
print("Val dist.  :", pretty_counts(val_counts, dm.idx_to_class))
print("Test dist. :", pretty_counts(test_counts, dm.idx_to_class))

# 4) Quick sanity-check del primer batch (shape y canales)
images, targets = next(iter(train_loader))
print("Batch shape:", tuple(images.shape), "| dtype:", images.dtype)
print("Targets shape:", tuple(targets.shape), "| unique labels in batch:", sorted(targets.unique().tolist()))

# Si cfg.input_mode == "replicate_to_3", deberías ver (B, 3, 224, 224).
# Si cfg.input_mode == "single_channel_conv", deberías ver (B, 1, 224, 224).


Sizes -> train: 5617 | val: 703 | test: 703 | total: 7023
Class mapping: {0: 'glioma', 1: 'healthy', 2: 'meningioma', 3: 'pituitary'}
Train dist.: {'glioma': 1297, 'healthy': 1600, 'meningioma': 1315, 'pituitary': 1405}
Val dist.  : {'glioma': 162, 'healthy': 200, 'meningioma': 165, 'pituitary': 176}
Test dist. : {'glioma': 162, 'healthy': 200, 'meningioma': 165, 'pituitary': 176}


d:\Users\Usuario\Documents\ENTROPY_LAB\maestria_ia\maia_lab\Deep_Learning\MicroProyecto1\myenv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
# ============================================================
# 🔍 Learning Rate Finder (LR Range Test)
# ============================================================

# ✅ 1. Crear subset de 500 imágenes del set de entrenamiento
subset_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(train_loader.dataset, range(0, min(500, len(train_loader.dataset)))),
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory
)

# ✅ 2. Ejecutar el test de LR
# Escanea LR desde 1e-6 hasta 5e-2 en 100 iteraciones
lrs, losses = lr_range_test(
    model,
    subset_loader,
    cfg,
    start_lr=1e-6,
    end_lr=5e-2,
    num_iters=100
)

# ✅ 3. Graficar resultados
plot_lr_finder(lrs, losses)

# ✅ 4. Sugerir LR óptimo automáticamente (opcional)
import numpy as np

losses_smooth = np.convolve(losses, np.ones(5)/5, mode='valid')
min_idx = np.argmin(losses_smooth)
suggested_lr = lrs[max(0, min_idx - 3)]  # punto antes del mínimo
print(f"📈 Sugerencia automática: usa un LR cercano a {suggested_lr:.2e}")

# Puedes actualizar cfg.base_lr antes de entrenar:
# cfg.base_lr = suggested_lr



## 10. **Main** (orquestación)
> No se ejecuta por defecto. Completa `cfg.data_dir` y descomenta para correr.


In [ ]:
def main(cfg: Config):
    dm = DataModule(cfg)
    dm.setup()
    train_loader, val_loader, test_loader = dm.loaders()

    builder = ModelBuilder(cfg)
    model = builder.build()

    # --- LR Finder (opcional) ---
    # subset_loader = torch.utils.data.DataLoader(
    #     torch.utils.data.Subset(train_loader.dataset, indices=range(0, min(512, len(train_loader.dataset)))),
    #     batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory
    # )
    # lrs, losses = lr_range_test(model, subset_loader, cfg, start_lr=1e-6, end_lr=5e-2, num_iters=150)
    # plot_lr_finder(lrs, losses)
    # cfg.base_lr = 1e-3  # ajustar según el gráfico

    trainer = Trainer(cfg)
    model, history = trainer.fit(model, train_loader, val_loader)

    # Evaluación final en TEST con el mejor checkpoint cargado por Trainer
    criterion = criterion_fn()
    test_loss, test_acc, test_f1 = trainer.validate(model, test_loader, criterion)
    print(f"TEST | loss={test_loss:.4f} | acc={test_acc:.4f} | macro_f1={test_f1:.4f}")

    # Guardar historia y métricas de test
    os.makedirs(cfg.out_dir, exist_ok=True)
    np.save(os.path.join(cfg.out_dir, f"{cfg.run_name}_history.npy"), history, allow_pickle=True)
    with open(os.path.join(cfg.out_dir, f"{cfg.run_name}_test_metrics.json"), "w") as f:
        f.write(json.dumps({"loss": test_loss, "acc": test_acc, "macro_f1": test_f1}, indent=2))

In [ ]:
if __name__ == "__main__":
    main(cfg)


## 11. Recomendación sobre **Learning Rate**
- Ejecuta el **LR Range Test** en un **subset** (p.ej., 512 muestras, 100–200 iteraciones).
- Elige un LR en la **zona estable** antes del disparo de la pérdida (o cerca del mínimo).  
- Para entrenamiento completo, dos alternativas robustas:
  1. **OneCycleLR** con `max_lr` elegido del LR Finder (con *warm-up* implícito y decay).  
  2. **ReduceLROnPlateau** monitorizando **macro-F1** (descender LR cuando no mejore).

> Como regla práctica inicial: `1e-3` con **AdamW** o `3e-2` con **SGD+momentum** suele ser buen punto de partida para fine-tuning de ResNet‑50 en 224×224; **afina** tras el LR Finder.

## 12. **Grayscale** y cambios en ResNet‑50
- **Opción A (por defecto):** *replicar* el canal gris a 3 → **no** modificas la arquitectura preentrenada y mantienes pesos RGB intactos.  
- **Opción B:** *modificar* `conv1` para `in_channels=1`, promediando pesos RGB. Equivale a aprender un filtro inicial específico para escala de grises.  
Ambas son válidas; mantenemos **A** por simplicidad y compatibilidad, con posibilidad de cambiar a **B** desde `cfg.input_mode`.



---
### Qué sigue
1. Validar `cfg.data_dir` y la estructura del dataset.  
2. (Opcional) Correr **LR Finder** y ajustar `cfg.base_lr` o `one_cycle_max_lr`.  
3. Ejecutar `main(cfg)` y revisar métricas/curvas.  
4. Añadir matriz de confusión y gráficos para el informe (máx. 2 páginas).

> Este esqueleto prioriza SRP/SOLID, con módulos **reusables** y **extensibles**, evitando duplicación (**DRY**) y con una jerarquía clara (**Atomic Design**).


Celda de diagnóstico

Esta celda:

Llama a DataModule.setup()

Imprime tamaños de train/val/test

Verifica que la suma coincide con el total

Muestra distribución por clase en cada split

Confirma forma del primer batch (para chequear 1 canal vs 3 canales según input_mode)


from collections import Counter
import numpy as np

# 1) Instanciar y preparar
dm = DataModule(cfg)
dm.setup()
train_loader, val_loader, test_loader = dm.loaders()

# 2) Tamaños por split
n_train = len(train_loader.dataset)
n_val   = len(val_loader.dataset)
n_test  = len(test_loader.dataset)
print(f"Sizes -> train: {n_train} | val: {n_val} | test: {n_test} | total: {n_train+n_val+n_test}")

# 3) Verificación de consistencia
#   Cada dataset es un Subset(ImageFolder), así que podemos acceder a sus indices y a targets del ImageFolder base.
def split_counts(subset):
    idxs = np.array(subset.indices)
    targets = np.array(subset.dataset.targets)
    labels = targets[idxs]
    return Counter(labels)

def pretty_counts(counter, idx_to_class):
    return { idx_to_class[k]: v for k, v in sorted(counter.items(), key=lambda kv: kv[0]) }

train_counts = split_counts(train_loader.dataset)
val_counts   = split_counts(val_loader.dataset)
test_counts  = split_counts(test_loader.dataset)

print("Class mapping:", dm.idx_to_class)
print("Train dist.:", pretty_counts(train_counts, dm.idx_to_class))
print("Val dist.  :", pretty_counts(val_counts, dm.idx_to_class))
print("Test dist. :", pretty_counts(test_counts, dm.idx_to_class))

# 4) Quick sanity-check del primer batch (shape y canales)
images, targets = next(iter(train_loader))
print("Batch shape:", tuple(images.shape), "| dtype:", images.dtype)
print("Targets shape:", tuple(targets.shape), "| unique labels in batch:", sorted(targets.unique().tolist()))

# Si cfg.input_mode == "replicate_to_3", deberías ver (B, 3, 224, 224).
# Si cfg.input_mode == "single_channel_conv", deberías ver (B, 1, 224, 224).


In [ ]:
dm = DataModule(cfg)
dm.setup()
train_loader, val_loader, test_loader = dm.loaders()

builder = ModelBuilder(cfg)
model = builder.build()


# ================================================
# 🛡️ LR Finder robusto (subset 500, 100 iteraciones)
# ================================================
import gc, torch, random, numpy as np

# 1) Medidas de seguridad memoria/proceso
try:
    torch.multiprocessing.set_sharing_strategy("file_system")
except Exception:
    pass

torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()

# 2) Construir subset aleatorio de 500 del TRAIN ya cargado
n = len(train_loader.dataset)
subset_size = min(500, n)
subset_indices = random.sample(range(n), subset_size)

safe_bs = min(cfg.batch_size, 16)  # batch más pequeño para LR test

subset_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(train_loader.dataset, subset_indices),
    batch_size=safe_bs,
    shuffle=True,
    num_workers=0,                 # << clave: sin multiproceso
    pin_memory=False,              # << clave: sin pin a pag. fija
    persistent_workers=False       # << clave: no mantener workers
)

# 3) Ejecutar LR Range Test (100 iteraciones)
lrs, losses = lr_range_test(
    model,
    subset_loader,
    cfg,
    start_lr=1e-6,
    end_lr=5e-2,
    num_iters=100
)

# 4) Graficar e imprimir sugerencia
plot_lr_finder(lrs, losses)

losses_smooth = np.convolve(losses, np.ones(5)/5, mode='valid')
min_idx = int(np.argmin(losses_smooth))
suggested_lr = lrs[max(0, min_idx - 3)]  # un poco antes del mínimo
print(f"📈 Sugerencia automática de LR ≈ {suggested_lr:.2e} (ajústalo en cfg.base_lr)")
